# Import Libraries

In [ ]:
!pip install pykan tensorflow torch scikit-learn matplotlib seaborn opencv-python

In [ ]:
!pip install tf-keras-vis lime

In [ ]:
import os
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, InceptionV3
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet_full
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception_full
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
import datetime
import torch
from kan import KAN
import matplotlib.pyplot
from sklearn.metrics import confusion_matrix

# Load Datset

In [ ]:

drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/PG/Fall 2025/CSE754/CSE754_Project/dataset"
BENIGN_US_FOLDER = os.path.join(ROOT, "Ultrasound images of benign thyroid lesions")
BENIGN_PATH_FOLDER = os.path.join(ROOT, "Cytological images of benign thyroid lesions")
MALIGNANT_US_FOLDER = os.path.join(ROOT, "Ultrasound images of papillary thyroid carcinoma")
MALIGNANT_PATH_FOLDER = os.path.join(ROOT, "Cytological images of papillary thyroid carcinoma")

In [ ]:

# Create a timestamped output directory
OUTPUT_DIR = os.path.join(ROOT, "outputs", datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"All outputs will be saved to: {OUTPUT_DIR}")

In [ ]:
def show_first_files(folder_path, label):
    print(f"\n CONTENTS OF: {label}")
    try:
        if os.path.exists(folder_path):
            files = sorted(os.listdir(folder_path))
            clean_files = [f for f in files if not f.startswith('.')]

            if len(clean_files) == 0:
                print("   [EMPTY FOLDER]")
            else:

                for i, f in enumerate(clean_files[:5]):
                    print(f"   File {i+1}: {f}")
        else:
            print(f"Folder not found at: {folder_path}")
    except Exception as e:
        print(f"Error: {e}")


show_first_files(BENIGN_US_FOLDER, "Benign Ultrasound")
show_first_files(BENIGN_PATH_FOLDER, "Benign Pathology")
show_first_files(MALIGNANT_US_FOLDER, "Malignant Ultrasound")
show_first_files(MALIGNANT_PATH_FOLDER, "Malignant Pathology")

**Configuration**

**Logic Matching**

In [ ]:

def load_complex_pairs(us_dir, path_dir, label_value):
    X_us = []
    X_path = []
    y = []

    print(f"Processing {os.path.basename(us_dir)}...")

    if not os.path.exists(us_dir) or not os.path.exists(path_dir):
        print("Folder missing.")
        return [], [], []

    us_map = {}
    for f in os.listdir(us_dir):
        if f.startswith('.'): continue
        pid = os.path.splitext(f)[0]
        us_map[pid] = f

    path_files = sorted(os.listdir(path_dir))
    matches = 0

    for f_path in path_files:
        if f_path.startswith('.'): continue

        pid = f_path.split('_')[0]

        if pid in us_map:
            try:
                f_us = us_map[pid]
                img_us = cv2.imread(os.path.join(us_dir, f_us))
                if img_us is None: continue
                img_us = cv2.resize(img_us, (224, 224))

                img_path = cv2.imread(os.path.join(path_dir, f_path))
                if img_path is None: continue
                img_path = cv2.resize(img_path, (299, 299))

                X_us.append(cv2.cvtColor(img_us, cv2.COLOR_BGR2RGB))
                X_path.append(cv2.cvtColor(img_path, cv2.COLOR_BGR2RGB))
                y.append(label_value)
                matches += 1

            except Exception as e:
                print(f"   Error on {pid}: {e}")

    print(f"Found {matches} pairs (Patches linked to US).")
    return X_us, X_path, y

**Executing Data Loading**

In [ ]:
print("--- Loading Data ---")
b_us, b_path, b_y = load_complex_pairs(BENIGN_US_FOLDER, BENIGN_PATH_FOLDER, 0)
m_us, m_path, m_y = load_complex_pairs(MALIGNANT_US_FOLDER, MALIGNANT_PATH_FOLDER, 1)

# Model Training

In [ ]:
if len(b_y) + len(m_y) > 0:
      X_us_raw = np.array(b_us + m_us)
      X_path_raw = np.array(b_path + m_path)
      y = np.array(b_y + m_y)

      print(f"\nSUCCESS! Total Training Pairs: {len(y)}")
      print("\nExtracting Features (ResNet + Inception)")
      us_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
      path_model = InceptionV3(weights='imagenet', include_top=False, pooling='avg')

      feat_us = us_model.predict(preprocess_resnet_full(X_us_raw.copy()), verbose=1)
      feat_path = path_model.predict(preprocess_inception_full(X_path_raw.copy()), verbose=1)

      feat_fused = np.concatenate([feat_us, feat_path], axis=1)

      X_train, X_test, y_train, y_test = train_test_split(feat_fused, y, test_size=0.2, random_state=42)

      print("\nTraining KAN Model...")

      device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

      dataset = {
          'train_input': torch.from_numpy(X_train).float().to(device),
          'test_input': torch.from_numpy(X_test).float().to(device),
          'train_label': torch.from_numpy(y_train).long().to(device),
          'test_label': torch.from_numpy(y_test).long().to(device)
      }

      model = KAN(width=[4096, 64, 2], grid=5, k=3, seed=42, device=device)

      def train_acc(): return torch.mean((torch.argmax(model(dataset['train_input']), dim=1) == dataset['train_label']).float())
      def test_acc(): return torch.mean((torch.argmax(model(dataset['test_input']), dim=1) == dataset['test_label']).float())

      results = model.fit(dataset, opt="Adam", steps=25, metrics=(train_acc, test_acc), loss_fn=torch.nn.CrossEntropyLoss())

      print(f"\nFinal Accuracy: {results['test_acc'][-1]:.4f}")

else:
    print("Still 0 matches. There might be a deeper issue with the file content types.")

# Evaluation & Comparison

**Evaluate KAN model**

In [ ]:
print("Evaluating KAN Model...")
model.eval()

with torch.no_grad():

    logits = model(dataset['test_input'])
    kan_preds = torch.argmax(logits, dim=1).cpu().numpy()

    kan_acc = accuracy_score(y_test, kan_preds)
    kan_p, kan_r, kan_f1, _ = precision_recall_fscore_support(y_test, kan_preds, average='weighted')
    metrics_kan = [kan_acc, kan_p, kan_r, kan_f1]

print(f"KAN Evaluation Complete. Accuracy: {kan_acc:.2%}")

print("\n--- Detailed KAN Report ---")

print(classification_report(y_test, kan_preds, target_names=['Benign', 'Malignant']))

**Training Fused MLP**

In [ ]:
print("   Training Baseline (Fused MLP) for comparison...")

clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=25, random_state=42)
clf.fit(X_train, y_train)
mlp_preds = clf.predict(X_test)

mlp_acc = accuracy_score(y_test, mlp_preds)
mlp_p, mlp_r, mlp_f1, _ = precision_recall_fscore_support(y_test, mlp_preds, average='weighted')
metrics_mlp = [mlp_acc, mlp_p, mlp_r, mlp_f1]

**Comparison Results**

In [ ]:
results_df = pd.DataFrame([
    metrics_mlp,
    metrics_kan
],
columns=["Accuracy", "Precision", "Recall", "F1-Score"],
index=["Standard MLP (Black Box)", "Proposed KAN (Ours)"])

print("\nFINAL RESULTS COMPARISON")
print(results_df.round(4))

**Visualization**

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, kan_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title(f"Proposed KAN Model\nAccuracy: {kan_acc:.2%}")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.subplot(1, 2, 2)
cm_mlp = confusion_matrix(y_test, mlp_preds)
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Oranges', xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title(f"Standard MLP Model\nAccuracy: {mlp_acc:.2%}")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:



plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, kan_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title(f"Proposed KAN Model\nAccuracy: {kan_acc:.2%}")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.subplot(1, 2, 2)
cm_mlp = confusion_matrix(y_test, mlp_preds)
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Oranges', xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title(f"Standard MLP Model\nAccuracy: {mlp_acc:.2%}")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), bbox_inches='tight')
plt.show()

## Interpretability: GRAD-CAM and LIME

To understand what features the convolutional neural networks (ResNet50 and InceptionV3) are focusing on, we can use interpretability techniques like GRAD-CAM and LIME. Although our final models (KAN and MLP) operate on extracted features, these techniques can show us which parts of the *original images* contribute most to the feature extraction, providing insights into the visual cues being learned.

For this demonstration, we'll apply GRAD-CAM and LIME to the full ResNet50 and InceptionV3 models (trained on ImageNet with their default classification heads) to visualize their attention on sample images. This helps understand the underlying feature extraction process.

In [ ]:
# --- GRAD-CAM Implementation ---
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, InceptionV3
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet_full
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception_full

# For Grad-CAM
from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

# For LIME
from lime import lime_image

def visualize_gradcam(model, image, preprocess_fn, title, layer_name=None):
    # Expand dimensions for model input
    input_image = np.expand_dims(preprocess_fn(image.copy()), axis=0)

    # Select the last convolutional layer dynamically if not provided
    if layer_name is None:
        for layer in reversed(model.layers):
            # Check if the layer is a convolutional layer
            if isinstance(layer, tf.keras.layers.Conv2D):
                layer_name = layer.name
                break

        if layer_name is None:
            print(f"Could not find a convolutional layer in {model.name}.")
            return

    # Create Gradcam object
    gradcam = Gradcam(model, model_modifier=ReplaceToLinear(), clone=False)

    # Get a dummy score to activate Grad-CAM
    pred_probs = model.predict(input_image)
    score = CategoricalScore(np.argmax(pred_probs[0]))

    # Generate heatmap
    cam = gradcam(score, input_image, penultimate_layer=layer_name)
    heatmap = cam[0]

    # Resize heatmap to original image size
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Superimpose heatmap on original image
    superimposed_img = cv2.addWeighted(cv2.cvtColor(image, cv2.COLOR_RGB2BGR), 0.6, heatmap, 0.4, 0)

    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')

print("Generating GRAD-CAM visualizations...")

# Load ResNet50 model for GRAD-CAM
resnet_model_full = ResNet50(weights='imagenet', include_top=True)

# Safely check if b_us has elements before accessing index 0
sample_us_image = b_us[0] if len(b_us) > 0 else None

if sample_us_image is not None:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_us_image)
    plt.title("Original US Image (Benign)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    visualize_gradcam(resnet_model_full, sample_us_image, preprocess_resnet_full, "ResNet50 Grad-CAM (US)")
    plt.tight_layout()
    plt.show()
else:
    print("No US images available for Grad-CAM.")

# Load InceptionV3 model for GRAD-CAM
inception_model_full = InceptionV3(weights='imagenet', include_top=True)

sample_path_image = m_path[0] if len(m_path) > 0 else None

if sample_path_image is not None:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_path_image)
    plt.title("Original Path Image (Malignant)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    visualize_gradcam(inception_model_full, sample_path_image, preprocess_inception_full, "InceptionV3 Grad-CAM (Path)")
    plt.tight_layout()
    plt.show()
else:
    print("No Pathology images available for Grad-CAM.")

In [ ]:
# --- LIME Implementation ---

print("Generating LIME visualizations...")

explainer = lime_image.LimeImageExplainer()

# Define a prediction function for LIME (using ResNet50 for US images)
def predict_proba_resnet(images):
    # LIME passes un-preprocessed images (0-1 range typically), need to resize and preprocess
    resized_images = np.array([cv2.resize(img, (224, 224)) for img in images])
    preprocessed_images = preprocess_resnet_full(resized_images * 255.0) # LIME typically sends 0-1, so scale back to 0-255
    return resnet_model_full.predict(preprocessed_images)

if sample_us_image is not None:
    # LIME expects images in [0, 1] range for explanation
    # Ensure sample_us_image is float and in [0, 1] if it's not already
    sample_us_image_float = sample_us_image.astype(np.float32) / 255.0

    explanation = explainer.explain_instance(
        sample_us_image_float,
        predict_proba_resnet,
        top_labels=1, hide_color=0, num_samples=1000 #can be 1000
    )

    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True, num_features=5, hide_rest=True
    )

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_us_image)
    plt.title("Original US Image (Benign)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    # Convert temp to float32 before passing to cv2.cvtColor
    plt.imshow(cv2.cvtColor(temp.astype(np.float32), cv2.COLOR_BGR2RGB))
    plt.title("ResNet50 LIME (US, Positive Only)")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "lime_resnet50_us.png"), bbox_inches='tight')
    plt.show()
else:
    print("No US images available for LIME.")

In [ ]:
# --- Functions for Quantitative Evaluation ---

def calculate_faithfulness_drop(model, original_image, preprocess_fn, explanation_map, target_class_idx, num_perturbations=10):
    # Flatten and sort explanation values to identify most important pixels
    flat_map = explanation_map.flatten()
    sorted_indices = np.argsort(flat_map)[::-1] # Descending order of importance

    original_pred = model.predict(np.expand_dims(preprocess_fn(original_image.copy()), axis=0))[0][target_class_idx]

    perturbation_drops = []
    for i in range(1, num_perturbations + 1):
        # Mask out top N% of important pixels
        mask_percentage = i / num_perturbations
        num_pixels_to_mask = int(len(sorted_indices) * mask_percentage)
        pixels_to_mask_indices = sorted_indices[:num_pixels_to_mask]

        perturbed_image = original_image.copy()
        # Convert flattened index back to 2D for masking
        rows, cols = np.unravel_index(pixels_to_mask_indices, explanation_map.shape)

        # Apply mask: set important pixels to mean pixel value to reduce information content
        mean_pixel_value = np.mean(original_image, axis=(0,1))
        perturbed_image[rows, cols] = mean_pixel_value

        perturbed_pred = model.predict(np.expand_dims(preprocess_fn(perturbed_image), axis=0))[0][target_class_idx]
        perturbation_drops.append(original_pred - perturbed_pred)

    # Average drop as a measure of faithfulness
    return np.mean(perturbation_drops)

def calculate_sparsity(explanation_map, threshold_ratio=0.5):
    # Calculate the proportion of pixels above a certain threshold (e.g., 50% of max importance)
    max_val = np.max(explanation_map)
    threshold = max_val * threshold_ratio
    sparse_area = np.sum(explanation_map >= threshold)
    total_area = explanation_map.size
    return sparse_area / total_area

print("Quantitative evaluation functions defined.")

# Prepare for saving quantitative results to a file
quantitative_results_filepath = os.path.join(OUTPUT_DIR, "quantitative_interpretability_results.txt")
with open(quantitative_results_filepath, "w") as f:
    f.write("Quantitative Interpretability Results:\n\n")

In [ ]:
# Re-generate Grad-CAM heatmap for the sample US image
if sample_us_image is not None:
    # Expand dimensions for model input
    input_image_us = np.expand_dims(preprocess_resnet_full(sample_us_image.copy()), axis=0)

    # Get ResNet50's top predicted class for faithfulness calculation
    pred_probs_us = resnet_model_full.predict(input_image_us)
    target_class_idx_us = np.argmax(pred_probs_us[0])

    # Find the last conv layer for Grad-CAM
    layer_name_us = None
    for layer in reversed(resnet_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_us = layer.name
            break

    if layer_name_us is not None:
        gradcam_us = Gradcam(resnet_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_us = CategoricalScore(target_class_idx_us)
        cam_us = gradcam_us(score_us, input_image_us, penultimate_layer=layer_name_us)
        heatmap_us = cam_us[0]
        heatmap_us_resized = cv2.resize(heatmap_us, (sample_us_image.shape[1], sample_us_image.shape[0]))

        # Calculate metrics
        faith_us_gradcam = calculate_faithfulness_drop(resnet_model_full, sample_us_image, preprocess_resnet_full, heatmap_us_resized, target_class_idx_us)
        sparsity_us_gradcam = calculate_sparsity(heatmap_us_resized)

        result_str = f"ResNet50 Grad-CAM (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_gradcam:.4f}\n"
        result_str += f"ResNet50 Grad-CAM (US Image) - Sparsity (Area >= 50% max): {sparsity_us_gradcam:.2%}\n\n"
        print(result_str)
        with open(quantitative_results_filepath, "a") as f:
            f.write(result_str)
    else:
        print("Could not find a convolutional layer for ResNet50.")
else:
    print("No US images available for Grad-CAM evaluation.")

In [ ]:
# Re-generate LIME explanation for the sample US image
if sample_us_image is not None:
    explainer_lime = lime_image.LimeImageExplainer()

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    # To get the true target class index for the model, predict again on the original image
    input_image_us_for_pred = np.expand_dims(preprocess_resnet_full(sample_us_image.copy()), axis=0)
    pred_probs_us_lime = resnet_model_full.predict(input_image_us_for_pred)
    target_class_idx_us_lime = np.argmax(pred_probs_us_lime[0])

    # LIME explanation (using the pre-generated explanation from cell 921988d1 if it ran)
    # If not, we need to re-run the explanation. For robustness, let's assume it was run.
    # If you run this cell independently, ensure the LIME explanation is generated first.

    # For LIME, we extract the mask and convert it to a heatmap-like array for consistent metric calculation
    # The 'mask' returned by LIME typically highlights relevant superpixels. We'll use this.
    # The explanation object from the previous LIME cell is needed here.

    # Note: LIME's explanation is on `sample_us_image_float` (0-1 range)
    # We'll use the mask from `explanation.get_image_and_mask` as our explanation_map

    # Re-run LIME explanation for robustness in this cell
    sample_us_image_float = sample_us_image.astype(np.float32) / 255.0
    explanation_lime = explainer_lime.explain_instance(
        sample_us_image_float,
        predict_proba_resnet,
        top_labels=1, hide_color=0, num_samples=1000 #can be 1000
    )

    _, mask_lime = explanation_lime.get_image_and_mask(
        explanation_lime.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False # Use hide_rest=False to get the full mask
    )

    # The mask_lime from explanation.get_image_and_mask is 0s and 1s or similar intensity. Normalize it.
    # For faithfulness, we need an importance map. The mask is already a good one.
    # Let's convert the mask to a grayscale image (single channel) for consistency with heatmap_us_resized
    mask_lime_single_channel = np.mean(mask_lime, axis=2) if len(mask_lime.shape) == 3 else mask_lime

    # For LIME faithfulness, we'll directly use the mask for perturbation
    # We need to adapt `calculate_faithfulness_drop` for binary masks or weighted masks from LIME

    # Simpler faithfulness for LIME: mask out the important areas and check prediction drop
    # Let's consider the mask as the explanation_map for faithfulness calculation

    # Ensure mask_lime_single_channel has values in [0, 1] for scaling important pixels
    faith_us_lime = calculate_faithfulness_drop(resnet_model_full, sample_us_image, preprocess_resnet_full, mask_lime_single_channel, target_class_idx_us_lime)
    sparsity_us_lime = calculate_sparsity(mask_lime_single_channel)

    result_str = f"ResNet50 LIME (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_lime:.4f}\n"
    result_str += f"ResNet50 LIME (US Image) - Sparsity (Area >= 50% max): {sparsity_us_lime:.2%}\n\n"
    print(result_str)
    with open(quantitative_results_filepath, "a") as f:
        f.write(result_str)
else:
    print("No US images available for LIME evaluation.")

In [ ]:
# Re-generate Grad-CAM heatmap for the sample Pathology image
if sample_path_image is not None:
    # Expand dimensions for model input
    input_image_path = np.expand_dims(preprocess_inception_full(sample_path_image.copy()), axis=0)

    # Get InceptionV3's top predicted class for faithfulness calculation
    pred_probs_path = inception_model_full.predict(input_image_path)
    target_class_idx_path = np.argmax(pred_probs_path[0])

    # Find the last conv layer for Grad-CAM
    layer_name_path = None
    for layer in reversed(inception_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_path = layer.name
            break

    if layer_name_path is not None:
        gradcam_path = Gradcam(inception_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_path = CategoricalScore(target_class_idx_path)
        cam_path = gradcam_path(score_path, input_image_path, penultimate_layer=layer_name_path)
        heatmap_path = cam_path[0]
        heatmap_path_resized = cv2.resize(heatmap_path, (sample_path_image.shape[1], sample_path_image.shape[0]))

        # Calculate metrics
        faith_path_gradcam = calculate_faithfulness_drop(inception_model_full, sample_path_image, preprocess_inception_full, heatmap_path_resized, target_class_idx_path)
        sparsity_path_gradcam = calculate_sparsity(heatmap_path_resized)

        result_str = f"InceptionV3 Grad-CAM (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_gradcam:.4f}\n"
        result_str += f"InceptionV3 Grad-CAM (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_gradcam:.2%}\n\n"
        print(result_str)
        with open(quantitative_results_filepath, "a") as f:
            f.write(result_str)
    else:
        print("Could not find a convolutional layer for InceptionV3.")
else:
    print("No Pathology images available for Grad-CAM evaluation.")

In [ ]:
# Re-generate Grad-CAM heatmap for the sample Pathology image using ResNet50
if sample_path_image is not None:
    # Resize the pathology image to 224x224 for ResNet50
    resized_path_image_for_resnet = cv2.resize(sample_path_image.copy(), (224, 224))

    # Expand dimensions for model input
    input_image_path_resnet = np.expand_dims(preprocess_resnet_full(resized_path_image_for_resnet), axis=0)

    # Get ResNet50's top predicted class for faithfulness calculation
    pred_probs_path_resnet = resnet_model_full.predict(input_image_path_resnet)
    target_class_idx_path_resnet = np.argmax(pred_probs_path_resnet[0])

    # Find the last conv layer for Grad-CAM in ResNet50
    layer_name_path_resnet = None
    for layer in reversed(resnet_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_path_resnet = layer.name
            break

    if layer_name_path_resnet is not None:
        gradcam_path_resnet = Gradcam(resnet_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_path_resnet = CategoricalScore(target_class_idx_path_resnet)
        cam_path_resnet = gradcam_path_resnet(score_path_resnet, input_image_path_resnet, penultimate_layer=layer_name_path_resnet)
        heatmap_path_resnet = cam_path_resnet[0]
        # Resize heatmap to match the dimensions of the image used for perturbation (224x224) for faithfulness calculation
        heatmap_path_resnet_resized = cv2.resize(heatmap_path_resnet, (resized_path_image_for_resnet.shape[1], resized_path_image_for_resnet.shape[0]))

        # Calculate metrics
        faith_path_gradcam_resnet = calculate_faithfulness_drop(resnet_model_full, resized_path_image_for_resnet, preprocess_resnet_full, heatmap_path_resnet_resized, target_class_idx_path_resnet)
        sparsity_path_gradcam_resnet = calculate_sparsity(heatmap_path_resnet_resized)

        result_str = f"ResNet50 Grad-CAM (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_gradcam_resnet:.4f}\n"
        result_str += f"ResNet50 Grad-CAM (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_gradcam_resnet:.2%}\n\n"
        print(result_str)
        with open(quantitative_results_filepath, "a") as f:
            f.write(result_str)
    else:
        print("Could not find a convolutional layer for ResNet50.")
else:
    print("No Pathology images available for ResNet50 Grad-CAM evaluation.")

In [ ]:
# Re-generate LIME explanation for the sample Pathology image using ResNet50
if sample_path_image is not None:
    explainer_lime_resnet_path = lime_image.LimeImageExplainer()

    # Define a prediction function for LIME (using ResNet50 for pathology images)
    def predict_proba_resnet_path(images):
        resized_images = np.array([cv2.resize(img, (224, 224)) for img in images])
        preprocessed_images = preprocess_resnet_full(resized_images * 255.0)
        return resnet_model_full.predict(preprocessed_images)

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    input_image_path_for_pred = np.expand_dims(preprocess_resnet_full(sample_path_image.copy()), axis=0)
    pred_probs_path_lime_resnet = resnet_model_full.predict(input_image_path_for_pred)
    target_class_idx_path_lime_resnet = np.argmax(pred_probs_path_lime_resnet[0])

    sample_path_image_float = sample_path_image.astype(np.float32) / 255.0
    explanation_lime_resnet_path = explainer_lime_resnet_path.explain_instance(
        sample_path_image_float,
        predict_proba_resnet_path,
        top_labels=1, hide_color=0, num_samples=10
    )

    _, mask_lime_resnet_path = explanation_lime_resnet_path.get_image_and_mask(
        explanation_lime_resnet_path.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False
    )

    mask_lime_resnet_path_single_channel = np.mean(mask_lime_resnet_path, axis=2) if len(mask_lime_resnet_path.shape) == 3 else mask_lime_resnet_path

    faith_path_lime_resnet = calculate_faithfulness_drop(resnet_model_full, sample_path_image, preprocess_resnet_full, mask_lime_resnet_path_single_channel, target_class_idx_path_lime_resnet)
    sparsity_path_lime_resnet = calculate_sparsity(mask_lime_resnet_path_single_channel)

    result_str = f"ResNet50 LIME (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_lime_resnet:.4f}\n"
    result_str += f"ResNet50 LIME (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_lime_resnet:.2%}\n\n"
    print(result_str)
    with open(quantitative_results_filepath, "a") as f:
        f.write(result_str)
else:
    print("No Pathology images available for ResNet50 LIME evaluation.")

In [ ]:
if sample_us_image is not None:
    # Resize the US image to 299x299 for InceptionV3
    resized_us_image_for_inception = cv2.resize(sample_us_image.copy(), (299, 299))

    # Expand dimensions for model input
    input_image_us_inception = np.expand_dims(preprocess_inception_full(resized_us_image_for_inception), axis=0)

    # Get InceptionV3's top predicted class for faithfulness calculation
    pred_probs_us_inception = inception_model_full.predict(input_image_us_inception)
    target_class_idx_us_inception = np.argmax(pred_probs_us_inception[0])

    # Find the last conv layer for Grad-CAM in InceptionV3
    layer_name_us_inception = None
    for layer in reversed(inception_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_us_inception = layer.name
            break

    if layer_name_us_inception is not None:
        gradcam_us_inception = Gradcam(inception_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_us_inception = CategoricalScore(target_class_idx_us_inception)
        cam_us_inception = gradcam_us_inception(score_us_inception, input_image_us_inception, penultimate_layer=layer_name_us_inception)
        heatmap_us_inception = cam_us_inception[0]
        # Resize heatmap to match the dimensions of the image used for perturbation (299x299) for faithfulness calculation
        heatmap_us_inception_resized = cv2.resize(heatmap_us_inception, (resized_us_image_for_inception.shape[1], resized_us_image_for_inception.shape[0]))

        # Calculate metrics
        faith_us_gradcam_inception = calculate_faithfulness_drop(inception_model_full, resized_us_image_for_inception, preprocess_inception_full, heatmap_us_inception_resized, target_class_idx_us_inception)
        sparsity_us_gradcam_inception = calculate_sparsity(heatmap_us_inception_resized)

        result_str = f"InceptionV3 Grad-CAM (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_gradcam_inception:.4f}\n"
        result_str += f"InceptionV3 Grad-CAM (US Image) - Sparsity (Area >= 50% max): {sparsity_us_gradcam_inception:.2%}\n\n"
        print(result_str)
        with open(quantitative_results_filepath, "a") as f:
            f.write(result_str)
    else:
        print("Could not find a convolutional layer for InceptionV3.")
else:
    print("No US images available for InceptionV3 Grad-CAM evaluation.")

In [ ]:
# Re-generate LIME explanation for the sample US image using InceptionV3
if sample_us_image is not None:
    explainer_lime_inception_us = lime_image.LimeImageExplainer()

    # Define a prediction function for LIME (using InceptionV3 for US images)
    def predict_proba_inception_us(images):
        resized_images = np.array([cv2.resize(img, (299, 299)) for img in images])
        preprocessed_images = preprocess_inception_full(resized_images * 255.0)
        return inception_model_full.predict(preprocessed_images)

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    input_image_us_for_pred = np.expand_dims(preprocess_inception_full(sample_us_image.copy()), axis=0)
    pred_probs_us_lime_inception = inception_model_full.predict(input_image_us_for_pred)
    target_class_idx_us_lime_inception = np.argmax(pred_probs_us_lime_inception[0])

    sample_us_image_float = sample_us_image.astype(np.float32) / 255.0
    explanation_lime_inception_us = explainer_lime_inception_us.explain_instance(
        sample_us_image_float,
        predict_proba_inception_us,
        top_labels=1, hide_color=0, num_samples=10
    )

    _, mask_lime_inception_us = explanation_lime_inception_us.get_image_and_mask(
        explanation_lime_inception_us.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False
    )

    mask_lime_inception_us_single_channel = np.mean(mask_lime_inception_us, axis=2) if len(mask_lime_inception_us.shape) == 3 else mask_lime_inception_us

    faith_us_lime_inception = calculate_faithfulness_drop(inception_model_full, sample_us_image, preprocess_inception_full, mask_lime_inception_us_single_channel, target_class_idx_us_lime_inception)
    sparsity_us_lime_inception = calculate_sparsity(mask_lime_inception_us_single_channel)

    result_str = f"InceptionV3 LIME (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_lime_inception:.4f}\n"
    result_str += f"InceptionV3 LIME (US Image) - Sparsity (Area >= 50% max): {sparsity_us_lime_inception:.2%}\n\n"
    print(result_str)
    with open(quantitative_results_filepath, "a") as f:
        f.write(result_str)
else:
    print("No US images available for InceptionV3 LIME evaluation.")

In [ ]:
# Re-generate LIME explanation for the sample Pathology image using InceptionV3
if sample_path_image is not None:
    explainer_lime_inception_path = lime_image.LimeImageExplainer()

    # Define a prediction function for LIME (using InceptionV3 for pathology images)
    def predict_proba_inception_path(images):
        resized_images = np.array([cv2.resize(img, (299, 299)) for img in images])
        preprocessed_images = preprocess_inception_full(resized_images * 255.0)
        return inception_model_full.predict(preprocessed_images)

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    input_image_path_for_pred = np.expand_dims(preprocess_inception_full(sample_path_image.copy()), axis=0)
    pred_probs_path_lime_inception = inception_model_full.predict(input_image_path_for_pred)
    target_class_idx_path_lime_inception = np.argmax(pred_probs_path_lime_inception[0])

    sample_path_image_float = sample_path_image.astype(np.float32) / 255.0
    explanation_lime_inception_path = explainer_lime_inception_path.explain_instance(
        sample_path_image_float,
        predict_proba_inception_path,
        top_labels=1, hide_color=0, num_samples=10
    )

    _, mask_lime_inception_path = explanation_lime_inception_path.get_image_and_mask(
        explanation_lime_inception_path.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False
    )

    mask_lime_inception_path_single_channel = np.mean(mask_lime_inception_path, axis=2) if len(mask_lime_inception_path.shape) == 3 else mask_lime_inception_path

    faith_path_lime_inception = calculate_faithfulness_drop(inception_model_full, sample_path_image, preprocess_inception_full, mask_lime_inception_path_single_channel, target_class_idx_path_lime_inception)
    sparsity_path_lime_inception = calculate_sparsity(mask_lime_inception_path_single_channel)

    result_str = f"InceptionV3 LIME (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_lime_inception:.4f}\n"
    result_str += f"InceptionV3 LIME (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_lime_inception:.2%}\n\n"
    print(result_str)
    with open(quantitative_results_filepath, "a") as f:
        f.write(result_str)
else:
    print("No Pathology images available for InceptionV3 LIME evaluation.")

In [ ]:
# --- LIME Implementation ---

print("Generating LIME visualizations...")

explainer = lime_image.LimeImageExplainer()

# Define a prediction function for LIME (using ResNet50 for US images)
def predict_proba_resnet(images):
    # LIME passes un-preprocessed images (0-1 range typically), need to resize and preprocess
    resized_images = np.array([cv2.resize(img, (224, 224)) for img in images])
    preprocessed_images = preprocess_resnet_full(resized_images * 255.0) # LIME typically sends 0-1, so scale back to 0-255
    return resnet_model_full.predict(preprocessed_images)

if sample_us_image is not None:
    # LIME expects images in [0, 1] range for explanation
    # Ensure sample_us_image is float and in [0, 1] if it's not already
    sample_us_image_float = sample_us_image.astype(np.float32) / 255.0

    explanation = explainer.explain_instance(
        sample_us_image_float,
        predict_proba_resnet,
        top_labels=1, hide_color=0, num_samples=1000 #can be 1000
    )

    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True, num_features=5, hide_rest=True
    )

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_us_image)
    plt.title("Original US Image (Benign)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    # Convert temp to float32 before passing to cv2.cvtColor
    plt.imshow(cv2.cvtColor(temp.astype(np.float32), cv2.COLOR_BGR2RGB))
    plt.title("ResNet50 LIME (US, Positive Only)")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No US images available for LIME.")

## Comparing Interpretability Results: GRAD-CAM, LIME, and KAN

Each of these techniques offers a different perspective on model interpretability:

*   **GRAD-CAM** and **LIME** are **local interpretability** methods. They explain *why a specific prediction was made for a specific input instance* by highlighting important regions in the input image.
    *   **GRAD-CAM** uses gradients flowing into the last convolutional layer to produce a coarse localization map, showing areas most influential for a specific class prediction. It's good for understanding what features a CNN *learned* to focus on.
    *   **LIME** (Local Interpretable Model-agnostic Explanations) perturbs the input locally and observes how the model's prediction changes. It identifies superpixels (contiguous regions of similar pixels) that are most important for the prediction. LIME is model-agnostic, meaning it can explain any black-box model.

*   **KAN (Koopman operator based Neural Networks)**, on the other hand, aims for **global interpretability**. Unlike black-box models, KANs are designed to explicitly represent underlying functions (e.g., $f(x,y) = \sin(x) + x^2 + \exp(y)$) through a network of univariate spline functions. This means KANs can potentially reveal the mathematical relationships between input features and output, making their decision-making process transparent globally.

### How to Compare:

1.  **Visual Alignment**: For a given image, observe if the salient regions highlighted by GRAD-CAM and LIME visually correspond to structures or patterns that you would intuitively associate with a benign or malignant diagnosis. For example, if a model identifies a specific texture or shape as indicative of malignancy, do the heatmaps confirm this visual focus?

2.  **Feature Importance (KAN vs. Saliency)**: KANs operate on the *features extracted* by ResNet50 and InceptionV3. While KAN provides global insights into how these *features* combine, GRAD-CAM and LIME explain which *pixels* contributed to those features.
    *   **Direct Comparison Challenge**: It's difficult to directly compare a pixel-level saliency map to the functional form of a KAN layer. KAN provides a mathematical relationship between abstract features, while saliency maps provide visual cues from the raw image.
    *   **Indirect Insights**: If KAN identifies certain features (e.g., related to texture from the US image or cell morphology from the pathology image) as highly influential, the saliency maps can help *visually ground* these abstract features by showing *where in the image* those textures or morphologies are located. This provides a link between the image and the KAN's functional understanding.

3.  **Consistency**: Are the explanations consistent across different images of the same class? Do benign images consistently show attention on certain benign-specific features, and malignant images on malignant-specific features, as per GRAD-CAM/LIME? This consistency can then be related back to KAN's global rules.

In essence, GRAD-CAM and LIME offer *local visual explanations*, helping us understand *what the CNN saw* at the pixel level. KANs provide *global functional explanations*, showing *how the extracted features are mathematically combined* for the final prediction. Together, they offer a more comprehensive understanding of the entire diagnostic pipeline.

## Quantitative Evaluation of Interpretability

While visual inspection of GRAD-CAM and LIME heatmaps provides qualitative insights, quantitative metrics offer a more objective way to assess the quality of explanations. We will focus on two key metrics:

### 1. Faithfulness (or Fidelity)
Faithfulness measures how well an explanation reflects the behavior of the model. A faithful explanation correctly highlights the input features that are truly influential to the model's prediction. A common way to evaluate this is by perturbing the input based on the explanation and observing the change in the model's output for the predicted class.

**Implementation**: We can calculate the 'Area Over the Perturbation Curve' (AOPC) or simply observe the drop in prediction probability when the most important features (as identified by the explanation) are removed or 'erased' from the input image. A larger drop indicates higher faithfulness, as the explanation correctly pointed to features that were critical for the prediction.

### 2. Sparsity
Sparsity refers to the conciseness of an explanation. A sparse explanation uses a minimal number of features to convey the most important information. For image-based explanations, this can relate to the proportion of the image highlighted by the explanation. A more sparse explanation (using fewer pixels/regions) is generally preferred if it still retains high faithfulness, as it's easier to interpret.

**Implementation**: We can measure the percentage of pixels or the area covered by the salient regions identified by GRAD-CAM or LIME. A smaller percentage indicates higher sparsity.

In [ ]:
# --- Functions for Quantitative Evaluation ---

def calculate_faithfulness_drop(model, original_image, preprocess_fn, explanation_map, target_class_idx, num_perturbations=10):
    # Flatten and sort explanation values to identify most important pixels
    flat_map = explanation_map.flatten()
    sorted_indices = np.argsort(flat_map)[::-1] # Descending order of importance

    original_pred = model.predict(np.expand_dims(preprocess_fn(original_image.copy()), axis=0))[0][target_class_idx]

    perturbation_drops = []
    for i in range(1, num_perturbations + 1):
        # Mask out top N% of important pixels
        mask_percentage = i / num_perturbations
        num_pixels_to_mask = int(len(sorted_indices) * mask_percentage)
        pixels_to_mask_indices = sorted_indices[:num_pixels_to_mask]

        perturbed_image = original_image.copy()
        # Convert flattened index back to 2D for masking
        rows, cols = np.unravel_index(pixels_to_mask_indices, explanation_map.shape)

        # Apply mask: set important pixels to mean pixel value to reduce information content
        mean_pixel_value = np.mean(original_image, axis=(0,1))
        perturbed_image[rows, cols] = mean_pixel_value

        perturbed_pred = model.predict(np.expand_dims(preprocess_fn(perturbed_image), axis=0))[0][target_class_idx]
        perturbation_drops.append(original_pred - perturbed_pred)

    # Average drop as a measure of faithfulness
    return np.mean(perturbation_drops)

def calculate_sparsity(explanation_map, threshold_ratio=0.5):
    # Calculate the proportion of pixels above a certain threshold (e.g., 50% of max importance)
    max_val = np.max(explanation_map)
    threshold = max_val * threshold_ratio
    sparse_area = np.sum(explanation_map >= threshold)
    total_area = explanation_map.size
    return sparse_area / total_area

print("Quantitative evaluation functions defined.")

### Evaluating ResNet50 with GRAD-CAM (US Image)

We will now apply the faithfulness and sparsity metrics to the Grad-CAM explanation for the sample US image using the ResNet50 model.

In [ ]:
# Re-generate Grad-CAM heatmap for the sample US image
if sample_us_image is not None:
    # Expand dimensions for model input
    input_image_us = np.expand_dims(preprocess_resnet_full(sample_us_image.copy()), axis=0)

    # Get ResNet50's top predicted class for faithfulness calculation
    pred_probs_us = resnet_model_full.predict(input_image_us)
    target_class_idx_us = np.argmax(pred_probs_us[0])

    # Find the last conv layer for Grad-CAM
    layer_name_us = None
    for layer in reversed(resnet_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_us = layer.name
            break

    if layer_name_us is not None:
        gradcam_us = Gradcam(resnet_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_us = CategoricalScore(target_class_idx_us)
        cam_us = gradcam_us(score_us, input_image_us, penultimate_layer=layer_name_us)
        heatmap_us = cam_us[0]
        heatmap_us_resized = cv2.resize(heatmap_us, (sample_us_image.shape[1], sample_us_image.shape[0]))

        # Calculate metrics
        faith_us_gradcam = calculate_faithfulness_drop(resnet_model_full, sample_us_image, preprocess_resnet_full, heatmap_us_resized, target_class_idx_us)
        sparsity_us_gradcam = calculate_sparsity(heatmap_us_resized)

        print(f"ResNet50 Grad-CAM (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_gradcam:.4f}")
        print(f"ResNet50 Grad-CAM (US Image) - Sparsity (Area >= 50% max): {sparsity_us_gradcam:.2%}")
    else:
        print("Could not find a convolutional layer for ResNet50.")
else:
    print("No US images available for Grad-CAM evaluation.")

### Evaluating ResNet50 with LIME (US Image)

Next, we will evaluate the LIME explanation for the same sample US image.

In [ ]:
# Re-generate LIME explanation for the sample US image
if sample_us_image is not None:
    explainer_lime = lime_image.LimeImageExplainer()

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    # To get the true target class index for the model, predict again on the original image
    input_image_us_for_pred = np.expand_dims(preprocess_resnet_full(sample_us_image.copy()), axis=0)
    pred_probs_us_lime = resnet_model_full.predict(input_image_us_for_pred)
    target_class_idx_us_lime = np.argmax(pred_probs_us_lime[0])

    # LIME explanation (using the pre-generated explanation from cell 921988d1 if it ran)
    # If not, we need to re-run the explanation. For robustness, let's assume it was run.
    # If you run this cell independently, ensure the LIME explanation is generated first.

    # For LIME, we extract the mask and convert it to a heatmap-like array for consistent metric calculation
    # The 'mask' returned by LIME typically highlights relevant superpixels. We'll use this.
    # The explanation object from the previous LIME cell is needed here.

    # Note: LIME's explanation is on `sample_us_image_float` (0-1 range)
    # We'll use the mask from `explanation.get_image_and_mask` as our explanation_map

    # Re-run LIME explanation for robustness in this cell
    sample_us_image_float = sample_us_image.astype(np.float32) / 255.0
    explanation_lime = explainer_lime.explain_instance(
        sample_us_image_float,
        predict_proba_resnet,
        top_labels=1, hide_color=0, num_samples=1000 #can be 1000
    )

    _, mask_lime = explanation_lime.get_image_and_mask(
        explanation_lime.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False # Use hide_rest=False to get the full mask
    )

    # The mask_lime from explanation.get_image_and_mask is 0s and 1s or similar intensity. Normalize it.
    # For faithfulness, we need an importance map. The mask is already a good one.
    # Let's convert the mask to a grayscale image (single channel) for consistency with heatmap_us_resized
    mask_lime_single_channel = np.mean(mask_lime, axis=2) if len(mask_lime.shape) == 3 else mask_lime

    # For LIME faithfulness, we'll directly use the mask for perturbation
    # We need to adapt `calculate_faithfulness_drop` for binary masks or weighted masks from LIME

    # Simpler faithfulness for LIME: mask out the important areas and check prediction drop
    # Let's consider the mask as the explanation_map for faithfulness calculation

    # Ensure mask_lime_single_channel has values in [0, 1] for scaling important pixels
    faith_us_lime = calculate_faithfulness_drop(resnet_model_full, sample_us_image, preprocess_resnet_full, mask_lime_single_channel, target_class_idx_us_lime)
    sparsity_us_lime = calculate_sparsity(mask_lime_single_channel)

    print(f"ResNet50 LIME (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_lime:.4f}")
    print(f"ResNet50 LIME (US Image) - Sparsity (Area >= 50% max): {sparsity_us_lime:.2%}")
else:
    print("No US images available for LIME evaluation.")

### Evaluating InceptionV3 with GRAD-CAM (Pathology Image)

Finally, let's evaluate the Grad-CAM explanation for the sample pathology image using the InceptionV3 model.

## Visualizing KAN Spline Functions for Top Features

KANs achieve interpretability by representing each connection as a univariate spline function. By visualizing these splines, particularly in the first layer, we can understand how individual input features are transformed before being combined. To focus on the most influential features, we can quantify their importance, for example, by computing the L1 norm of the splines associated with each input feature. A larger L1 norm generally indicates a more significant contribution to the model's output.

In [ ]:
print("Calculating feature importance and visualizing KAN splines...")

# Calculate L1 norm for each input feature's connections in the first layer
l1_norms = []
num_input_features = model.width[0][0] if isinstance(model.width[0], list) or isinstance(model.width[0], tuple) else model.width[0] # Number of features in the input layer
num_hidden_neurons = model.width[1][0] if isinstance(model.width[1], list) or isinstance(model.width[1], tuple) else model.width[1] # Number of neurons in the first hidden layer

for i in range(num_input_features): # Iterate through each input feature
    feature_l1_sum = 0.0
    for j in range(num_hidden_neurons): # Iterate through connections to each hidden neuron
        # Correctly access the spline coefficients from the KANLayer
        spline_coeffs = model.act_fun[0].coef[i][j]
        feature_l1_sum += torch.sum(torch.abs(spline_coeffs)).item() # Sum L1 norm of spline coefficients
    l1_norms.append(feature_l1_sum)

# Get top N features based on their aggregated L1 norm
top_n_features_to_plot = 10 # Let's visualize the top 10 features
top_feature_indices = np.argsort(l1_norms)[::-1][:top_n_features_to_plot]

print(f"Top {top_n_features_to_plot} features based on L1 norm: {top_feature_indices.tolist()}")

# Plot the spline functions for these top features
plt.figure(figsize=(18, 2.5 * top_n_features_to_plot)) # Adjust figure size dynamically
model.plot(in_vars=top_feature_indices.tolist()) # Removed 'scale_in_var=True'
plt.suptitle(f"KAN Spline Functions for Top {top_n_features_to_plot} Input Features (L1-norm based)", y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
print("Calculating feature importance and visualizing KAN splines...")

# Calculate L1 norm for each input feature's connections in the first layer
l1_norms = []
num_input_features = model.width[0][0] if isinstance(model.width[0], list) or isinstance(model.width[0], tuple) else model.width[0] # Number of features in the input layer
num_hidden_neurons = model.width[1][0] if isinstance(model.width[1], list) or isinstance(model.width[1], tuple) else model.width[1] # Number of neurons in the first hidden layer

for i in range(num_input_features): # Iterate through each input feature
    feature_l1_sum = 0.0
    for j in range(num_hidden_neurons): # Iterate through connections to each hidden neuron
        # Correctly access the spline coefficients from the KANLayer
        spline_coeffs = model.act_fun[0].coef[i][j]
        feature_l1_sum += torch.sum(torch.abs(spline_coeffs)).item() # Sum L1 norm of spline coefficients
    l1_norms.append(feature_l1_sum)

# Get top N features based on their aggregated L1 norm
top_n_features_to_plot = 10 # Let's visualize the top 10 features
top_feature_indices = np.argsort(l1_norms)[::-1][:top_n_features_to_plot]

print(f"Top {top_n_features_to_plot} features based on L1 norm: {top_feature_indices.tolist()}")

# Plot the spline functions for these top features
# plt.figure(figsize=(18, 2.5 * top_n_features_to_plot)) # Adjust figure size dynamically
# model.plot(in_vars=top_feature_indices.tolist()) # Removed 'scale_in_var=True'
# plt.suptitle(f"KAN Spline Functions for Top {top_n_features_to_plot} Input Features (L1-norm based)", y=1.00)
# plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, "kan_spline_functions.png"), bbox_inches='tight')
# plt.show()

## Exploratory Data Analysis (EDA)

Let's begin by examining the distribution of our target variable (benign vs. malignant) and then look into the distributions of some of the most influential features identified by the KAN model. This will provide insights into the balance of our dataset and the characteristics of the features that contribute most to the model's predictions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of Target Classes (0: Benign, 1: Malignant)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['Benign', 'Malignant'])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette='viridis')
plt.title('Distribution of Target Classes (0: Benign, 1: Malignant)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['Benign', 'Malignant'])
plt.savefig(os.path.join(OUTPUT_DIR, "target_class_distribution.png"), bbox_inches='tight')
plt.show()

Next, we'll visualize the distributions of a few of the top features as determined by their L1 norms from the KAN model. This helps understand how these crucial features are distributed across the benign and malignant classes.

In [ ]:
num_plots = min(4, len(top_feature_indices)) # Plot up to 4 top features

plt.figure(figsize=(15, 4 * num_plots))

for i in range(num_plots):
    feature_idx = top_feature_indices[i]
    feature_values = feat_fused[:, feature_idx]

    # Create a temporary DataFrame for easier plotting with seaborn
    temp_df = pd.DataFrame({"Feature Value": feature_values, "Class": y})

    plt.subplot(num_plots, 2, 2*i + 1)
    sns.histplot(data=temp_df, x="Feature Value", hue="Class", kde=True, palette='coolwarm')
    plt.title(f'Distribution of KAN Top Feature {feature_idx} (Histogram)')
    plt.xlabel(f'Feature {feature_idx} Value')
    plt.ylabel('Count')

    plt.subplot(num_plots, 2, 2*i + 2)
    sns.boxplot(data=temp_df, x="Class", y="Feature Value", palette='coolwarm')
    plt.title(f'Box Plot of KAN Top Feature {feature_idx} by Class')
    plt.xlabel('Class (0: Benign, 1: Malignant)')
    plt.ylabel(f'Feature {feature_idx} Value')

plt.tight_layout()
plt.show()

In [ ]:
num_plots = min(4, len(top_feature_indices)) # Plot up to 4 top features

plt.figure(figsize=(15, 4 * num_plots))

for i in range(num_plots):
    feature_idx = top_feature_indices[i]
    feature_values = feat_fused[:, feature_idx]

    # Create a temporary DataFrame for easier plotting with seaborn
    temp_df = pd.DataFrame({"Feature Value": feature_values, "Class": y})

    plt.subplot(num_plots, 2, 2*i + 1)
    sns.histplot(data=temp_df, x="Feature Value", hue="Class", kde=True, palette='coolwarm')
    plt.title(f'Distribution of KAN Top Feature {feature_idx} (Histogram)')
    plt.xlabel(f'Feature {feature_idx} Value')
    plt.ylabel('Count')

    plt.subplot(num_plots, 2, 2*i + 2)
    sns.boxplot(data=temp_df, x="Class", y="Feature Value", palette='coolwarm')
    plt.title(f'Box Plot of KAN Top Feature {feature_idx} by Class')
    plt.xlabel('Class (0: Benign, 1: Malignant)')
    plt.ylabel(f'Feature {feature_idx} Value')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "kan_top_feature_distributions.png"), bbox_inches='tight')
plt.show()

### t-SNE Visualization of Fused Features

To further explore the data distribution and the separability of benign and malignant cases in the high-dimensional feature space, we will use t-distributed Stochastic Neighbor Embedding (t-SNE). t-SNE is a dimensionality reduction technique particularly well-suited for visualizing high-dimensional datasets in 2D or 3D, preserving local structures. This visualization will help us understand if the features extracted by ResNet50 and InceptionV3 create distinct clusters for each class.

In [ ]:
from sklearn.manifold import TSNE
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Check if feat_fused and y are defined. If not, raise a more informative error.
if 'feat_fused' not in globals() or 'y' not in globals():
    raise NameError("Variables 'feat_fused' and 'y' are not defined. Please ensure the 'Model Training' section (specifically the feature extraction cell) has been executed to generate these variables.")

print("Performing t-SNE dimensionality reduction...")

# Perform t-SNE on the fused features
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
tsne_results = tsne.fit_transform(feat_fused)

# Create a DataFrame for easier plotting
tsne_df = pd.DataFrame(data=tsne_results, columns=['t-SNE_1', 't-SNE_2'])
tsne_df['Class'] = y

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='t-SNE_1', y='t-SNE_2', hue='Class',
    palette=sns.color_palette('hsv', 2),
    data=tsne_df,
    legend='full',
    alpha=0.7
)
plt.title('t-SNE Visualization of Fused Features (Colored by Class)')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.show()

print("t-SNE visualization complete.")

In [ ]:
from sklearn.manifold import TSNE
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Check if feat_fused and y are defined. If not, raise a more informative error.
if 'feat_fused' not in globals() or 'y' not in globals():
    raise NameError("Variables 'feat_fused' and 'y' are not defined. Please ensure the 'Model Training' section (specifically the feature extraction cell) has been executed to generate these variables.")

print("Performing t-SNE dimensionality reduction...")

# Perform t-SNE on the fused features
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
tsne_results = tsne.fit_transform(feat_fused)

# Create a DataFrame for easier plotting
tsne_df = pd.DataFrame(data=tsne_results, columns=['t-SNE_1', 't-SNE_2'])
tsne_df['Class'] = y

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='t-SNE_1', y='t-SNE_2', hue='Class',
    palette=sns.color_palette('hsv', 2),
    data=tsne_df,
    legend='full',
    alpha=0.7
)
plt.title('t-SNE Visualization of Fused Features (Colored by Class)')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.savefig(os.path.join(OUTPUT_DIR, "tsne_visualization.png"), bbox_inches='tight')
plt.show()

print("t-SNE visualization complete.")

In [ ]:
# Re-generate Grad-CAM heatmap for the sample Pathology image
if sample_path_image is not None:
    # Expand dimensions for model input
    input_image_path = np.expand_dims(preprocess_inception_full(sample_path_image.copy()), axis=0)

    # Get InceptionV3's top predicted class for faithfulness calculation
    pred_probs_path = inception_model_full.predict(input_image_path)
    target_class_idx_path = np.argmax(pred_probs_path[0])

    # Find the last conv layer for Grad-CAM
    layer_name_path = None
    for layer in reversed(inception_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_path = layer.name
            break

    if layer_name_path is not None:
        gradcam_path = Gradcam(inception_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_path = CategoricalScore(target_class_idx_path)
        cam_path = gradcam_path(score_path, input_image_path, penultimate_layer=layer_name_path)
        heatmap_path = cam_path[0]
        heatmap_path_resized = cv2.resize(heatmap_path, (sample_path_image.shape[1], sample_path_image.shape[0]))

        # Calculate metrics
        faith_path_gradcam = calculate_faithfulness_drop(inception_model_full, sample_path_image, preprocess_inception_full, heatmap_path_resized, target_class_idx_path)
        sparsity_path_gradcam = calculate_sparsity(heatmap_path_resized)

        print(f"InceptionV3 Grad-CAM (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_gradcam:.4f}")
        print(f"InceptionV3 Grad-CAM (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_gradcam:.2%}")
    else:
        print("Could not find a convolutional layer for InceptionV3.")
else:
    print("No Pathology images available for Grad-CAM evaluation.")

### Evaluating ResNet50 with GRAD-CAM (Pathology Image)

Although ResNet50 is typically used for US images in this pipeline, we will evaluate its Grad-CAM explanation on a pathology image as requested, to assess its interpretability in a cross-modal context.

In [ ]:
# Re-generate Grad-CAM heatmap for the sample Pathology image using ResNet50
if sample_path_image is not None:
    # Resize the pathology image to 224x224 for ResNet50
    resized_path_image_for_resnet = cv2.resize(sample_path_image.copy(), (224, 224))

    # Expand dimensions for model input
    input_image_path_resnet = np.expand_dims(preprocess_resnet_full(resized_path_image_for_resnet), axis=0)

    # Get ResNet50's top predicted class for faithfulness calculation
    pred_probs_path_resnet = resnet_model_full.predict(input_image_path_resnet)
    target_class_idx_path_resnet = np.argmax(pred_probs_path_resnet[0])

    # Find the last conv layer for Grad-CAM in ResNet50
    layer_name_path_resnet = None
    for layer in reversed(resnet_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_path_resnet = layer.name
            break

    if layer_name_path_resnet is not None:
        gradcam_path_resnet = Gradcam(resnet_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_path_resnet = CategoricalScore(target_class_idx_path_resnet)
        cam_path_resnet = gradcam_path_resnet(score_path_resnet, input_image_path_resnet, penultimate_layer=layer_name_path_resnet)
        heatmap_path_resnet = cam_path_resnet[0]
        # Resize heatmap to match the dimensions of the image used for perturbation (224x224) for faithfulness calculation
        heatmap_path_resnet_resized = cv2.resize(heatmap_path_resnet, (resized_path_image_for_resnet.shape[1], resized_path_image_for_resnet.shape[0]))

        # Calculate metrics
        faith_path_gradcam_resnet = calculate_faithfulness_drop(resnet_model_full, resized_path_image_for_resnet, preprocess_resnet_full, heatmap_path_resnet_resized, target_class_idx_path_resnet)
        sparsity_path_gradcam_resnet = calculate_sparsity(heatmap_path_resnet_resized)

        print(f"ResNet50 Grad-CAM (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_gradcam_resnet:.4f}")
        print(f"ResNet50 Grad-CAM (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_gradcam_resnet:.2%}")
    else:
        print("Could not find a convolutional layer for ResNet50.")
else:
    print("No Pathology images available for ResNet50 Grad-CAM evaluation.")

### Evaluating ResNet50 with LIME (Pathology Image)

In [ ]:
# Re-generate LIME explanation for the sample Pathology image using ResNet50
if sample_path_image is not None:
    explainer_lime_resnet_path = lime_image.LimeImageExplainer()

    # Define a prediction function for LIME (using ResNet50 for pathology images)
    def predict_proba_resnet_path(images):
        resized_images = np.array([cv2.resize(img, (224, 224)) for img in images])
        preprocessed_images = preprocess_resnet_full(resized_images * 255.0)
        return resnet_model_full.predict(preprocessed_images)

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    input_image_path_for_pred = np.expand_dims(preprocess_resnet_full(sample_path_image.copy()), axis=0)
    pred_probs_path_lime_resnet = resnet_model_full.predict(input_image_path_for_pred)
    target_class_idx_path_lime_resnet = np.argmax(pred_probs_path_lime_resnet[0])

    sample_path_image_float = sample_path_image.astype(np.float32) / 255.0
    explanation_lime_resnet_path = explainer_lime_resnet_path.explain_instance(
        sample_path_image_float,
        predict_proba_resnet_path,
        top_labels=1, hide_color=0, num_samples=10
    )

    _, mask_lime_resnet_path = explanation_lime_resnet_path.get_image_and_mask(
        explanation_lime_resnet_path.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False
    )

    mask_lime_resnet_path_single_channel = np.mean(mask_lime_resnet_path, axis=2) if len(mask_lime_resnet_path.shape) == 3 else mask_lime_resnet_path

    faith_path_lime_resnet = calculate_faithfulness_drop(resnet_model_full, sample_path_image, preprocess_resnet_full, mask_lime_resnet_path_single_channel, target_class_idx_path_lime_resnet)
    sparsity_path_lime_resnet = calculate_sparsity(mask_lime_resnet_path_single_channel)

    print(f"ResNet50 LIME (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_lime_resnet:.4f}")
    print(f"ResNet50 LIME (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_lime_resnet:.2%}")
else:
    print("No Pathology images available for ResNet50 LIME evaluation.")

### Evaluating InceptionV3 with GRAD-CAM (US Image)

Similar to the previous cross-modal evaluation, we will assess InceptionV3's Grad-CAM explanation on a US image.

In [ ]:
if sample_us_image is not None:
    # Resize the US image to 299x299 for InceptionV3
    resized_us_image_for_inception = cv2.resize(sample_us_image.copy(), (299, 299))

    # Expand dimensions for model input
    input_image_us_inception = np.expand_dims(preprocess_inception_full(resized_us_image_for_inception), axis=0)

    # Get InceptionV3's top predicted class for faithfulness calculation
    pred_probs_us_inception = inception_model_full.predict(input_image_us_inception)
    target_class_idx_us_inception = np.argmax(pred_probs_us_inception[0])

    # Find the last conv layer for Grad-CAM in InceptionV3
    layer_name_us_inception = None
    for layer in reversed(inception_model_full.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            layer_name_us_inception = layer.name
            break

    if layer_name_us_inception is not None:
        gradcam_us_inception = Gradcam(inception_model_full, model_modifier=ReplaceToLinear(), clone=False)
        score_us_inception = CategoricalScore(target_class_idx_us_inception)
        cam_us_inception = gradcam_us_inception(score_us_inception, input_image_us_inception, penultimate_layer=layer_name_us_inception)
        heatmap_us_inception = cam_us_inception[0]
        # Resize heatmap to match the dimensions of the image used for perturbation (299x299) for faithfulness calculation
        heatmap_us_inception_resized = cv2.resize(heatmap_us_inception, (resized_us_image_for_inception.shape[1], resized_us_image_for_inception.shape[0]))

        # Calculate metrics
        faith_us_gradcam_inception = calculate_faithfulness_drop(inception_model_full, resized_us_image_for_inception, preprocess_inception_full, heatmap_us_inception_resized, target_class_idx_us_inception)
        sparsity_us_gradcam_inception = calculate_sparsity(heatmap_us_inception_resized)

        print(f"InceptionV3 Grad-CAM (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_gradcam_inception:.4f}")
        print(f"InceptionV3 Grad-CAM (US Image) - Sparsity (Area >= 50% max): {sparsity_us_gradcam_inception:.2%}")
    else:
        print("Could not find a convolutional layer for InceptionV3.")
else:
    print("No US images available for InceptionV3 Grad-CAM evaluation.")

### Evaluating InceptionV3 with LIME (US Image)

In [ ]:
# Re-generate LIME explanation for the sample US image using InceptionV3
if sample_us_image is not None:
    explainer_lime_inception_us = lime_image.LimeImageExplainer()

    # Define a prediction function for LIME (using InceptionV3 for US images)
    def predict_proba_inception_us(images):
        resized_images = np.array([cv2.resize(img, (299, 299)) for img in images])
        preprocessed_images = preprocess_inception_full(resized_images * 255.0)
        return inception_model_full.predict(preprocessed_images)

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    input_image_us_for_pred = np.expand_dims(preprocess_inception_full(sample_us_image.copy()), axis=0)
    pred_probs_us_lime_inception = inception_model_full.predict(input_image_us_for_pred)
    target_class_idx_us_lime_inception = np.argmax(pred_probs_us_lime_inception[0])

    sample_us_image_float = sample_us_image.astype(np.float32) / 255.0
    explanation_lime_inception_us = explainer_lime_inception_us.explain_instance(
        sample_us_image_float,
        predict_proba_inception_us,
        top_labels=1, hide_color=0, num_samples=10
    )

    _, mask_lime_inception_us = explanation_lime_inception_us.get_image_and_mask(
        explanation_lime_inception_us.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False
    )

    mask_lime_inception_us_single_channel = np.mean(mask_lime_inception_us, axis=2) if len(mask_lime_inception_us.shape) == 3 else mask_lime_inception_us

    faith_us_lime_inception = calculate_faithfulness_drop(inception_model_full, sample_us_image, preprocess_inception_full, mask_lime_inception_us_single_channel, target_class_idx_us_lime_inception)
    sparsity_us_lime_inception = calculate_sparsity(mask_lime_inception_us_single_channel)

    print(f"InceptionV3 LIME (US Image) - Faithfulness (Avg. Prob Drop): {faith_us_lime_inception:.4f}")
    print(f"InceptionV3 LIME (US Image) - Sparsity (Area >= 50% max): {sparsity_us_lime_inception:.2%}")
else:
    print("No US images available for InceptionV3 LIME evaluation.")

### Evaluating InceptionV3 with LIME (Pathology Image)

This evaluates the LIME explanation for the pathology image using the InceptionV3 model.

In [ ]:
# Re-generate LIME explanation for the sample Pathology image using InceptionV3
if sample_path_image is not None:
    explainer_lime_inception_path = lime_image.LimeImageExplainer()

    # Define a prediction function for LIME (using InceptionV3 for pathology images)
    def predict_proba_inception_path(images):
        resized_images = np.array([cv2.resize(img, (299, 299)) for img in images])
        preprocessed_images = preprocess_inception_full(resized_images * 255.0)
        return inception_model_full.predict(preprocessed_images)

    # We need the class ID that LIME explained, which is `explanation.top_labels[0]`
    input_image_path_for_pred = np.expand_dims(preprocess_inception_full(sample_path_image.copy()), axis=0)
    pred_probs_path_lime_inception = inception_model_full.predict(input_image_path_for_pred)
    target_class_idx_path_lime_inception = np.argmax(pred_probs_path_lime_inception[0])

    sample_path_image_float = sample_path_image.astype(np.float32) / 255.0
    explanation_lime_inception_path = explainer_lime_inception_path.explain_instance(
        sample_path_image_float,
        predict_proba_inception_path,
        top_labels=1, hide_color=0, num_samples=10
    )

    _, mask_lime_inception_path = explanation_lime_inception_path.get_image_and_mask(
        explanation_lime_inception_path.top_labels[0],
        positive_only=True, num_features=5, hide_rest=False
    )

    mask_lime_inception_path_single_channel = np.mean(mask_lime_inception_path, axis=2) if len(mask_lime_inception_path.shape) == 3 else mask_lime_inception_path

    faith_path_lime_inception = calculate_faithfulness_drop(inception_model_full, sample_path_image, preprocess_inception_full, mask_lime_inception_path_single_channel, target_class_idx_path_lime_inception)
    sparsity_path_lime_inception = calculate_sparsity(mask_lime_inception_path_single_channel)

    print(f"InceptionV3 LIME (Path Image) - Faithfulness (Avg. Prob Drop): {faith_path_lime_inception:.4f}")
    print(f"InceptionV3 LIME (Path Image) - Sparsity (Area >= 50% max): {sparsity_path_lime_inception:.2%}")
else:
    print("No Pathology images available for InceptionV3 LIME evaluation.")

## Visualizing Grad-CAM for Images with High Activation of Top KAN Features

To bridge the gap between KAN's abstract feature importance and visual interpretability, we will now identify images from our dataset that maximally activate each of the KAN's top features. For these selected images, we will then generate and display their Grad-CAM heatmaps. This approach helps us understand what visual patterns in the original images correspond to the highly influential features processed by the KAN model.

In [ ]:
print("Generating Grad-CAM for images activating top KAN features...")

top_n_features_to_plot = 10 # Define top_n_features_to_plot for this cell
n_us_features = 2048 # ResNet50 output size (before KAN)
n_path_features = 2048 # InceptionV3 output size (before KAN)

plt.figure(figsize=(15, 5 * top_n_features_to_plot)) # Adjust figure size dynamically

for i, feature_idx in enumerate(top_feature_indices):
    original_image = None
    model_for_gradcam = None
    preprocess_fn_for_gradcam = None
    title_prefix = f"KAN Top Feature {feature_idx}: "

    if feature_idx < n_us_features:
        # This feature comes from the ResNet50 (US image) branch
        feature_values_for_this_kan_feature = feat_us[:, feature_idx]
        sample_idx_in_raw_data = np.argmax(feature_values_for_this_kan_feature)
        original_image = X_us_raw[sample_idx_in_raw_data]
        model_for_gradcam = resnet_model_full
        preprocess_fn_for_gradcam = preprocess_resnet_full
        image_type = "US Image"

    else:
        # This feature comes from the InceptionV3 (Pathology image) branch
        feature_idx_in_path_features = feature_idx - n_us_features
        feature_values_for_this_kan_feature = feat_path[:, feature_idx_in_path_features]
        sample_idx_in_raw_data = np.argmax(feature_values_for_this_kan_feature)
        original_image = X_path_raw[sample_idx_in_raw_data]
        model_for_gradcam = inception_model_full
        preprocess_fn_for_gradcam = preprocess_inception_full
        image_type = "Path Image"

    if original_image is not None:
        plt.subplot(top_n_features_to_plot, 2, 2*i + 1)
        plt.imshow(original_image)
        plt.title(f"{title_prefix} Original {image_type}")
        plt.axis('off')

        plt.subplot(top_n_features_to_plot, 2, 2*i + 2)
        visualize_gradcam(model_for_gradcam, original_image, preprocess_fn_for_gradcam, f"{title_prefix} Grad-CAM ({image_type})")
    else:
        print(f"Could not find image for KAN top feature {feature_idx}")

plt.tight_layout()
plt.show()

In [ ]:
print("Generating Grad-CAM for images activating top KAN features...")

top_n_features_to_plot = 10 # Define top_n_features_to_plot for this cell
n_us_features = 2048 # ResNet50 output size (before KAN)
n_path_features = 2048 # InceptionV3 output size (before KAN)

plt.figure(figsize=(15, 5 * top_n_features_to_plot)) # Adjust figure size dynamically

for i, feature_idx in enumerate(top_feature_indices):
    original_image = None
    model_for_gradcam = None
    preprocess_fn_for_gradcam = None
    title_prefix = f"KAN Top Feature {feature_idx}: "

    if feature_idx < n_us_features:
        # This feature comes from the ResNet50 (US image) branch
        feature_values_for_this_kan_feature = feat_us[:, feature_idx]
        sample_idx_in_raw_data = np.argmax(feature_values_for_this_kan_feature)
        original_image = X_us_raw[sample_idx_in_raw_data]
        model_for_gradcam = resnet_model_full
        preprocess_fn_for_gradcam = preprocess_resnet_full
        image_type = "US Image"

    else:
        # This feature comes from the InceptionV3 (Pathology image) branch
        feature_idx_in_path_features = feature_idx - n_us_features
        feature_values_for_this_kan_feature = feat_path[:, feature_idx_in_path_features]
        sample_idx_in_raw_data = np.argmax(feature_values_for_this_kan_feature)
        original_image = X_path_raw[sample_idx_in_raw_data]
        model_for_gradcam = inception_model_full
        preprocess_fn_for_gradcam = preprocess_inception_full
        image_type = "Path Image"

    if original_image is not None:
        plt.subplot(top_n_features_to_plot, 2, 2*i + 1)
        plt.imshow(original_image)
        plt.title(f"{title_prefix} Original {image_type}")
        plt.axis('off')

        plt.subplot(top_n_features_to_plot, 2, 2*i + 2)
        visualize_gradcam(model_for_gradcam, original_image, preprocess_fn_for_gradcam, f"{title_prefix} Grad-CAM ({image_type})")
    else:
        print(f"Could not find image for KAN top feature {feature_idx}")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "gradcam_top_kan_features.png"), bbox_inches='tight')
plt.show()